# Middleware

### It provides a way to more tightly control what happens inside the agent. Middleware is useful for following:

- Tracking agent behavior with logging, analytics and debugging.
- Transforming prompt, tool selection, and output formatting.
- Adding retries, fallback, and early termination logic.
- Applying rate limit, guardrails and PII detection.

In [41]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [42]:
### Summarization Middleware: Automatic summarize conversation history when approaching token limit.

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

In [43]:
# Applying trigger based on Message length

agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    tools = [],
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger = ("messages", 10), # Trigger summarization when there are more than 10 messages in the conversation history
            keep = ("messages", 4) # Keep the last 4 messages in the conversation history after summarization
        )
    ]
)

In [44]:
## Run with a thread ID

config = {"configurable": {"thread_id": "test-thread"}}

In [45]:
# Alternative test data
questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 1000-500?",
    "what is 2^10?",
    "what is 3*5?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='14dc3bd2-d8a1-4821-8db6-2fbaf4ad054d'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "what is 2+2?"\n2.  **Identify Core Task:** This is a basic arithmetic question.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely.\n5.  **Check for Accuracy/Context:** The answer is universally accepted as 4. No tricks or hidden meanings detected.\n6.  **Output Generation:** "2 + 2 equals 4." (or simply "4")\n\nI\'ll keep it direct and accurate.✅\n</think>\n\n2 + 2 equals **4**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 17, 'total_tokens': 168, 'completion_time': 0.287351536, 'completion_tokens_details': None, 'prompt_time': 0.000939713, 'prompt_tokens_details': None, 'queue_time': 0.062761642, 'total_time': 0.288291249}

### Token Length

In [46]:
# Applying trigger based on Token length

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long responses to use more tokens"""
    return f"Found hotels in {city}"

agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    tools = [search_hotel],
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger = ("tokens", 550), # Trigger summarization when token count exceeds 550
            keep = ("tokens", 200) # Keep the last 200 tokens after summarization
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

def count_tokens(messages):
    total_char = sum(len(str(m.content)) for m in messages)
    return total_char // 4  # Rough estimate: 1 token ~ 4 characters

In [47]:
# Run test

cities = ["New York", "Los Angeles", "Houston", "Phoenix", "San Antonio"]

for city in cities:
    response = agent.invoke({"messages":[HumanMessage(content=f"Search hotels in {city}")]}, config)
    tokens = count_tokens(response['messages'])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

New York: ~118 tokens, 4 messages
[HumanMessage(content='Search hotels in New York', additional_kwargs={}, response_metadata={}, id='85197409-99dd-45f7-b7fb-6e507157aa9e'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to search for hotels in New York.\nI have a tool `search_hotel` that takes a `city` parameter.\nI need to call this tool with "New York" as the city.\n\nPlan:\n1. Call `search_hotel` with `city`="New York".\n2. Present the results to the user.\n\nStep 1: Call the tool.\nTool: search_hotel\nArguments: {"city": "New York"}\n\nThis matches the user\'s request directly. No further reasoning needed.\n', 'tool_calls': [{'id': 'z9rvthg8q', 'function': {'arguments': '{"city":"New York"}', 'name': 'search_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 278, 'total_tokens': 423, 'completion_time': 0.295956132, 'completion_tokens_details': {'reasoning_tokens': 115}, 'prompt_time': 0.022519

### Fraction

In [50]:
# Applying trigger based on Token length

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long responses to use more tokens"""
    return f"Found hotels in {city}"

agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    tools = [search_hotel],
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger = ("fraction", 0.005), # Trigger summarization when token count exceeds 0.5% of the model's max token limit
            keep = ("fraction", 0.002) # Keep the last 0.2% of the model's max token limit after summarization
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4  # Rough estimate: 1 token ~ 4 characters

# Run test

cities = ["Paris", "London", "Tokyo", "Dubai", "Sydney"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Hotels in {city}")]}, 
        config=config)
    tokens = count_tokens(response['messages'])
    fraction = tokens / 360  # Assuming the model's max token limit is 36,000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} messages")
    print(f"{response['messages']}")

ValueError: Model profile information is required to use fractional token limits, and is unavailable for the specified model. Please use absolute token counts instead, or pass `

ChatModel(..., profile={"max_input_tokens": ...})`.

with a desired integer value of the model's maximum input tokens.

# Human in the loop

### It pasues the exceution for human approval, editing or rejecting of tool calls before they execute. It is useful for the following:

- Compliance workflow where human oversight is mandatory
- Long running conversation where human feedback guide the agent

In [49]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by it's ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to ID: {recipient} with subject: {subject}"

In [51]:
agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":{
                    "allowed_decisions": ["approve", "reject", "edit"]
                },
                "read_email_tool":False,
                }
        )
    ]
)

In [52]:
config = {"configurable": {"thread_id": "test-approve"}}

#Setup-1
result = agent.invoke({"messages":[HumanMessage(content="Send email to test@example.com with subject 'Greet' and body 'How are you?'")]}, 
                      config=config)

In [53]:
result

{'messages': [HumanMessage(content="Send email to test@example.com with subject 'Greet' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='053e7a4f-06ff-42f1-89fe-fc9f8129ef74'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email.\nI need to use the `send_email_tool` function.\nThe recipient is 'test@example.com'.\nThe subject is 'Greet'.\nThe body is 'How are you?'.\nI have all the required parameters.\nI will call the `send_email_tool` with these values.\n", 'tool_calls': [{'id': '1t4n65bgq', 'function': {'arguments': '{"body":"How are you?","recipient":"test@example.com","subject":"Greet"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 376, 'total_tokens': 507, 'completion_time': 0.257440131, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.030862778, 'prompt_tokens_details': None, 'queue_time': 0.033156502, '

In [54]:
#Step-2: Approve
from langgraph.types import Command
if "__interrupt__" in result:
    print(" Paused...... Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result["messages"][-1].content}")

 Paused...... Approving...
Result: The email has been successfully sent to test@example.com with the subject 'Greet' and body 'How are you?'.


In [55]:
result

{'messages': [HumanMessage(content="Send email to test@example.com with subject 'Greet' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='053e7a4f-06ff-42f1-89fe-fc9f8129ef74'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email.\nI need to use the `send_email_tool` function.\nThe recipient is 'test@example.com'.\nThe subject is 'Greet'.\nThe body is 'How are you?'.\nI have all the required parameters.\nI will call the `send_email_tool` with these values.\n", 'tool_calls': [{'id': '1t4n65bgq', 'function': {'arguments': '{"body":"How are you?","recipient":"test@example.com","subject":"Greet"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 376, 'total_tokens': 507, 'completion_time': 0.257440131, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.030862778, 'prompt_tokens_details': None, 'queue_time': 0.033156502, '

# Reject

In [56]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by it's ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to ID: {recipient} with subject: {subject}"

agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":{
                    "allowed_decisions": ["approve", "reject", "edit"]
                },
                "read_email_tool":False,
                }
        )
    ]
)

In [57]:
config = {"configurable": {"thread_id": "test-reject"}}

#Setup-1
result = agent.invoke({"messages":[HumanMessage(content="Send email to test@example.com with subject 'Greet' and body 'How are you?'")]}, 
                      config=config)

In [58]:
#Step-2: Reject
from langgraph.types import Command
if "__interrupt__" in result:
    print(" Paused...... Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result["messages"][-1].content}")

 Paused...... Approving...
Result: I attempted to send the email, but the action was rejected. Please check your email settings or try again.


# Editing

In [92]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [93]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [94]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='f292662d-9487-48b2-a71e-a454a4684387'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email.\nI have a tool `send_email_tool` that can be used for this purpose.\nThe tool requires three parameters: `recipient`, `subject`, and `body`.\nThe user has provided all of these:\n- recipient: 'wrong@email.com'\n- subject: 'Test'\n- body: 'Hello'\n\nI will call the `send_email_tool` with these parameters.\n", 'tool_calls': [{'id': '4p87gs85f', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 146, 'prompt_tokens': 372, 'total_tokens': 518, 'completion_time': 0.286145957, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0.031711046, '

In [95]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: The email was sent to `correct@email.com` with the subject 'Corrected Subject' and body 'This was edited by human before sending'. Would you like me to send the email to `wrong@email.com` with the subject 'Test' and body 'Hello' as originally requested?
